# Use Cases & Updating the Scene

**Part I · Visualization** — Tutorial 02

A decision guide for *how* you put objects into a scene and update them over
time. `pytanga.viz` offers several equivalent ways to add or modify entities —
`add()`, the `viz(...)` / `viz.new(...)` call shorthand, and context managers —
and several ways to push changes to the viewer (`show()`, `display()`, `flush()`).
Each has a sweet spot, so the final section gives a "which method to use when"
table.

> **Prerequisites:** the [quick tour](../01_quick_tour/) and the `Visualizer`
> basics in [Tutorial 04](../04_getting_started/).


## Setup


In [ ]:
import math

from pytanga.geometry import Direction, Line, Point, Sphere
from pytanga.viz import SphereStyle, Visualizer


## 1. Three ways to add objects

- `viz.add(entity, ...)` places an entity in the main scene and returns its
  entity id string — use it later with `update_entity()` / `update_style()` / `remove()`.
- `viz(entity, ...)` (or `viz.new(entity, ...)`) does the same but returns a
  `VizObjectRef`, a mutable handle with `.entity`, `.style`, and transform helpers.
- `with viz:` (or `with viz.scene("name"):`) clears the scene and shows it on
  entry, then flushes on exit.


In [ ]:
viz = Visualizer(title="Tanga — add() vs call")

# add() returns a plain entity id string
pid = viz.add(Point(1, 2, 3), color="#ff4444", label="$P$")
print("add() ->", pid)

# viz(...) / viz.new(...) returns a mutable VizObjectRef
ref = viz(Point(4, 5, 6), color="#44ff44", label="$Q$")
print("viz(...) ->", type(ref).__name__)

viz.display_snapshot()


## 2. Context managers

A context manager resets the scene, calls `show()`, and flushes on exit — no
server bookkeeping to remember. `with viz.scene("name")` does the same for a
named scene.


In [ ]:
with Visualizer() as viz:  # clear + show on entry, flush on exit
    viz(Point(1, 2, 3), color="#ff4444")
    viz(Sphere(Point(0, 0, 0), radius=2.5), style=SphereStyle(wireframe=True), opacity=0.3)
    viz(Line(origin=Point(0, 0, 0), direction=Direction(1, 0, 0)), color="#44ff44")


## 3. Updating the scene step-by-step

`flush()` pushes the current scene state to every connected viewer. `show()` and
`display()` are **idempotent within a cell**: re-running a cell does **not** open
a second viewer — it flushes the latest state into the already-open one. The
viewer is keyed by an optional `viewer_name`, the notebook cell id, or the scene
name.


In [ ]:
viz = Visualizer()
viz(Point(1, 2, 3), color="#ff4444")
viz.show()          # opens the inline viewer (starts the server)

viz(Point(4, 5, 6), color="#44ff44")
viz.show()          # no new viewer — just flushes the update


## 4. In-place updates vs fresh objects

For animation there are two strategies:

- **Pre-create + update in place** — create objects once with `viz(...)` and
  update `ref.entity` each frame. Only changed entities are pushed; the mesh
  transform, parent, and style are preserved. This is the performant choice.
- **Fresh objects per frame** — pass `auto_clear=True` to `animate()`; each frame
  flushes, then removes the objects added after the loop began (anything added
  *before* the loop persists). Concise, but less performant.

See [Tutorial 13](../13_animation/) for the full animation reference.


In [ ]:
# Pre-create + update in place
viz = Visualizer()
viz.show()
p = viz(Point(3, 0, 0), color="#ff4444")

angle = 0.0
for dt in viz.animate(fps=30):
    angle += 3.0 * dt
    p.entity = Point(3 * math.cos(angle), 3 * math.sin(angle), 0)
    viz.flush()
    if angle > 2 * math.pi:
        break


In [ ]:
# Fresh objects per frame with auto_clear
viz = Visualizer()
viz.show()
viz(Point(0, 0, 0), color="#ffffff")  # persists across frames

angle = 0.0
for dt in viz.animate(fps=30, auto_clear=True):
    angle += 3.0 * dt
    viz(Point(3 * math.cos(angle), 3 * math.sin(angle), 0), color="#ff4444")
    viz.flush()
    if angle > 2 * math.pi:
        break


## 5. Which method to use when

**Python script**

| Situation | Recommended |
|---|---|
| One-off demo, no animation | context manager (`with viz: …`) |
| One-off demo, animation | `animate(auto_clear=True)` for quick, short scripts |
| Long-running, no animation | build the scene, then `show()` + `wait()` |
| Long-running, animation | pre-create with `viz(...)` and update `.entity` in place |
| Interactive | `VisualizerApp` |
| Static snapshot | `viz.export_snapshot("scene.html")` |
| Animation recording | `start_animation_recording()` + animated HTML export |

**Jupyter notebook**

| Situation | Recommended |
|---|---|
| One-off demo, no animation | context manager |
| One-off demo, animation | `animate(auto_clear=True)` |
| Long-running, no animation | idempotent `show()` / `display()` re-renders |
| Long-running, animation | pre-create with `viz(...)` and update `.entity` in place |
| Interactive | `VisualizerApp` |
| Static snapshot | `viz.display_snapshot()` (embedded inline) |
| Animation recording | record a loop + animated HTML export |


## Visual Examples

A standalone HTML figure built step-by-step, exported via `export_snapshot()`.


In [ ]:
viz = Visualizer(title="Tanga — Step-by-step scene")
viz.add(Point(1, 2, 3), color="#ff4444", label="$P$")
viz.add(
    Sphere(Point(0, 0, 0), 2.0),
    style=SphereStyle(wireframe=True),
    opacity=0.3,
    label="$S$",
)
viz.add(Line(origin=Point(0, 0, 0), direction=Direction(1, 0, 0)), color="#44ff44")
viz.export_snapshot("_output/02_use_cases_figure.html", overwrite=True)
print("figure written")


## Summary

| Task | API |
|---|---|
| Add, get an id | `viz.add(entity, ...)` |
| Add, get a mutable handle | `viz(entity, ...)` / `viz.new(entity, ...)` |
| Clear + show, flush on exit | `with viz:` / `with viz.scene("name"):` |
| Push changes | `viz.flush()` |
| Re-render without a second viewer | repeated `viz.show()` / `viz.display()` |
| Update in place | `ref.entity = ...` / `viz.update_entity(...)` / `viz.update_style(...)` |
| Per-frame adds | `viz.animate(auto_clear=True)` |

**Next:** [03 — Jupyter Notebooks](../03_jupyter/) or
[04 — Getting Started](../04_getting_started/).
